# 02 · Modelling: seasonal LME competition (RHR / HRV)

Fits the six-stage, BIC-selected linear mixed-effects model competition on the cleaned
monthly cohort (`dummy_data[_hrv].csv` in the public input schema — see
[`cohort_io.py`](../src/cohort_io.py)) and writes every model artefact to
`results/{run_tag}/`. Run it once per metric (`METRIC` in `CONFIG`); the model-derived
tables and figures are then regenerated from these files by
[`03_model_results.ipynb`](03_model_results.ipynb).

| Output | File (under `results/{run_tag}/`) |
| :--- | :--- |
| Run configuration / thinning record | `run_config.json`, `thinning_summary.json`, `thinning_dropped_user_ids.csv` |
| Per-model audit (coefficients, covariance, VIF, validation CSV/SVGs, random effects) | `<timestamp>_<formula>/…` incl. `model-row.csv` |
| All candidate rows (one per fitted model) | `lme-models-results-{metric}.csv` |
| Overall winner (REML refit) | `model_{metric}_winner.pkl`, `model_{metric}_coefficients_interpreted.csv` |
| OLS temperature-residualisation parameters | `ols_temperature_params.csv` |
| Stage checkpoint (resume support) | `checkpoint_state.json` |

**Method.** Stage 1 baseline (age × sex) → 2 seasonality (harmonics vs day length)
→ 3 seasonal × day-length interactions (diagnostic: fitted and reported, not carried
forward) → 4 spatial (latitude ± longitude, ± seasonal interaction)
→ 5 temperature (mean vs max/min, ± seasonal interaction) → 6 joint competition with
temperature **residualised** on latitude/season (OLS) to break collinearity.
Winners are chosen by BIC (ML fits); the overall winner is refitted with REML for reporting.
The progression lives in [`stage_competition.py`](../src/stage_competition.py) (unit-tested
with a mock fitter). `STAGE_OVERRIDES` in `CONFIG` forces a different candidate to progress
(e.g. `{'Stage1': '1.1_Linear'}`) for counterfactual paths; the results CSV then records
`bic_winner` / `progressed` / `forced` / `on_path` per row.

The fast random-intercept fitter lives in [`lme_binned.py`](../src/lme_binned.py);
modelling helpers in [`seasonality_lib.py`](../src/seasonality_lib.py). File access goes
through [`file_io.py`](../src/file_io.py) (`./data` or `$WPM_DATA_DIR`).

**Resuming.** Every completed model writes `model-row.csv` into its folder and every
completed stage updates `checkpoint_state.json`; re-running the notebook with the same
`CONFIG` skips finished work. A run tag is tied to one data configuration: the notebook
refuses to continue when `run_config.json` (metric, thinning, subsample fraction, seed)
or the checkpoint (target, candidate spec, overrides, sample size) was written for
different settings. Set `FORCE_CLEAN = True` to start over.

The UN-region and country trend plots require at least 500 users per group
(`MIN_USERS_PER_GROUP` in `CONFIG`; lower it for smaller cohorts).

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import gc
import io
import json
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # library modules (public release)
import cohort_io as cio
import file_io as fio
import lme_binned
import paper_tables
import seasonality_lib as sl
import stage_competition

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
METRIC = 'rhr'  # 'rhr' | 'hrv'
THIN_RHR = True  # density-based thinning of the age-22 artefact (RHR only)
RHR_RUN_TAG = 'rhr_rerun'  # results/ sub-folder of the RHR run
HRV_RUN_TAG = 'hrv_rerun'  # results/ sub-folder of the HRV run
SUBSAMPLE_FRAC = 0.10  # fraction of users used for fitting (1.0 = everyone)
MAX_STAGE = 6  # stop after this stage (1-6)
# Counterfactual progression: force a candidate (or an interaction decision)
# to be carried forward regardless of BIC, e.g. {'Stage1': '1.1_Linear'} or
# {'Stage4_Interaction': False}. {} = pure BIC selection.
# Decision points: Stage1, Stage2, Stage4_Base, Stage4_Interaction,
# Stage5_Base, Stage5_Interaction, Stage6 (see stage_competition.py; Stage 3
# is diagnostic only and has no decision point).
STAGE_OVERRIDES = {}
GENERATE_PLOTS = False  # full diagnostic plot set for *every* candidate (slow)
FORCE_CLEAN = False  # ignore existing checkpoint / completed models
RANDOM_STATE = 42
# UN regions / countries with fewer distinct users are left out of the grouped
# trend plots; lower it for small cohorts.
MIN_USERS_PER_GROUP = 500

METRIC_CFG = {
    'rhr': dict(
        colname='avg_resting_heart_rate_bpm',
        units='bpm',
        title='Resting Heart Rate',
        summary_range=(30.0, 150.0),
    ),
    'hrv': dict(
        colname='avg_hrv',
        units='ms',
        title='Heart Rate Variability',
        summary_range=(5.0, 250.0),
    ),
}
cfg = METRIC_CFG[METRIC]
colname, units, colname_title = cfg['colname'], cfg['units'], cfg['title']
# HRV is modelled on the log scale (coefficients are % changes).
colname_c = 'target_log' if METRIC == 'hrv' else colname
thinned = THIN_RHR and METRIC == 'rhr'
run_tag = RHR_RUN_TAG if METRIC == 'rhr' else HRV_RUN_TAG
STAGE_OVERRIDES = stage_competition.validate_overrides(STAGE_OVERRIDES)
if STAGE_OVERRIDES:
  print(
      f'STAGE_OVERRIDES active: {STAGE_OVERRIDES} -> use a dedicated run tag!'
  )

data_dir = fio.get_data_dir()
extra_dir = fio.get_extra_dir(data_dir)
results_dir = fio.get_results_dir(run_tag, data_dir)
fio.make_dirs(results_dir)
run_config = dict(
    metric=METRIC,
    thin_rhr=thinned,
    subsample_frac=SUBSAMPLE_FRAC,
    max_stage=MAX_STAGE,
    stage_overrides=STAGE_OVERRIDES,
    random_state=RANDOM_STATE,
    started=time.strftime('%Y-%m-%d %H:%M:%S'),
)
# A run tag belongs to one data configuration: refuse to resume (or to reuse
# completed models) when the settings that define the fitted sample differ.
run_config_path = os.path.join(results_dir, 'run_config.json')
if fio.file_exists(run_config_path) and not FORCE_CLEAN:
  with fio.open_file(run_config_path) as f:
    previous = json.load(f)
  data_keys = ('metric', 'thin_rhr', 'subsample_frac', 'random_state')
  changed = {
      k: (previous.get(k), run_config[k])
      for k in data_keys
      if previous.get(k) != run_config[k]
  }
  if changed:
    raise ValueError(
        f'results/{run_tag} was produced with different data settings'
        f' {changed} (previous, current); set FORCE_CLEAN = True or use'
        ' another run tag.'
    )
with fio.open_file(run_config_path, 'w') as f:
  json.dump(run_config, f, indent=2)
print(
    f'{METRIC.upper()}  thinned={thinned}  frac={SUBSAMPLE_FRAC}  ->'
    f' {results_dir}'
)

In [ ]:
NEEDED_COLS = [
    'user_id',
    'activity_period',
    'gender',
    'age',
    'lat_e7',
    'lng_e7',
    colname,
    'un_region',
    'country_2_alpha_cd',
    'dl_monthly',
    'avg_temp_mean',
    'avg_temp_min',
    'avg_temp_max',
]

t0 = time.time()
# cohort_io maps the public CSV headers and units to the column names used by
# the code.
input_path = cio.resolve_input_path(data_dir, METRIC)
frhr = cio.load_cohort(data_dir, METRIC, columns=NEEDED_COLS, path=input_path)
print(
    f'Loaded {len(frhr):,} user-months / {frhr["user_id"].nunique():,} users'
    f' from {input_path} in {time.time() - t0:.0f}s'
)

In [ ]:
# ----------------------------------------------------------------------------
# Thinning of the age-22 false-birth-year artefact
# Must happen BEFORE subsampling and modelling. The cohort tables / figures
# describing the thinned cohort are produced by 01_cohort_stats.ipynb.
# ----------------------------------------------------------------------------
if thinned:
  frhr, thin_stats = sl.apply_density_based_thinning(
      frhr,
      target_col=colname,
      val_min=cfg['summary_range'][0],
      val_max=cfg['summary_range'][1],
      random_state=RANDOM_STATE,
  )
  with fio.open_file(
      os.path.join(results_dir, 'thinning_dropped_user_ids.csv'), 'w'
  ) as f:
    pd.Series(sorted(thin_stats['dropped_user_ids']), name='user_id').to_csv(
        f, index=False
    )
  with fio.open_file(
      os.path.join(results_dir, 'thinning_summary.json'), 'w'
  ) as f:
    json.dump(
        {k: v for k, v in thin_stats.items() if isinstance(v, (int, float))},
        f,
        indent=2,
    )
  print(
      f'After thinning: {len(frhr):,} user-months /'
      f' {frhr["user_id"].nunique():,} users'
  )

In [ ]:
# ----------------------------------------------------------------------------
# Feature engineering (shared implementation: seasonality_lib, also used by
# publication_figures.py so figures and models see identical predictors)
# ----------------------------------------------------------------------------
frhr['age_group'] = pd.cut(
    frhr['age'], bins=np.arange(18, 90, 10), include_lowest=True
)
frhr['lat_group'] = pd.cut(
    frhr['lat_e7'] / 1e7, bins=[-60, -30, 0, 30, 60, 90], include_lowest=True
)

# Sex: centred binary (Male = 1) so the intercept is the population mean;
# |latitude|; longitude on the circle.
male_share = sl.add_sex_and_longitude_features(frhr)

# Continuous predictors: Gelman standardisation (centre, divide by 2 SD) of the
# variable and of its square, computed on the FULL cohort (before subsampling).
# The statistics are kept to back-transform coefficients and are written out
# (feature_stats.json, below) so that 03_model_results.ipynb rebuilds the
# predictors exactly as fitted.
feature_summary = sl.standardise_model_features(frhr)
scaling_factors = sl.scaling_factors_from_summary(feature_summary)

# Seasonal harmonics (annual + semi-annual).
sl.add_month_harmonics(frhr)

if METRIC == 'hrv':
  frhr['target_log'] = np.log(frhr[colname])
optimal_lambda = (
    0 if METRIC == 'hrv' else None
)  # log-transform flag for sl plotting

## LME model competition

In [ ]:
# ----------------------------------------------------------------------------
# User subsample for fitting (users with >= 2 months), compact dtypes
# ----------------------------------------------------------------------------
frhr['user_id_n'] = frhr['user_id'].astype('category').cat.codes
unique_users = frhr['user_id_n'].unique()
if SUBSAMPLE_FRAC < 1.0:
  sampled = pd.Series(unique_users).sample(
      frac=SUBSAMPLE_FRAC, random_state=RANDOM_STATE
  )
  frhr_subsample = frhr[frhr['user_id_n'].isin(sampled)].copy()
else:
  frhr_subsample = frhr
del frhr  # the full frame is no longer needed (re-run from the load cell to get it back)
gc.collect()

user_counts = frhr_subsample['user_id_n'].value_counts()
frhr_subsample = frhr_subsample[
    frhr_subsample['user_id_n'].isin(user_counts[user_counts > 1].index)
].copy()
user_mapping_df = frhr_subsample[['user_id_n', 'user_id']].drop_duplicates()
frhr_subsample = frhr_subsample.drop(columns=['user_id'])
float_cols = frhr_subsample.select_dtypes(include=['float64']).columns
frhr_subsample[float_cols] = frhr_subsample[float_cols].astype('float32')
print(
    f'Fitting sample: {frhr_subsample["user_id_n"].nunique():,} users,'
    f' {len(frhr_subsample):,} rows'
)

# ----------------------------------------------------------------------------
# Temperature residualised on latitude x season (orthogonalisation, Stage 6)
# with the OLS in sl.TEMP_RESIDUAL_FORMULA; its parameters are written out for
# the table tab:{metric}_ols_params.
# ----------------------------------------------------------------------------
ols_info = sl.residualise_temperature(frhr_subsample, method='harmonic')
for temp_var, info in ols_info.items():
  frhr_subsample[f'{temp_var}_res'] = frhr_subsample[f'{temp_var}_res'].astype(
      'float32'
  )
  print(f'  {temp_var}_res  (OLS R2 = {info["rsquared"]:.4f})')
for col in ['avg_temp_max_sq_c_res', 'avg_temp_min_sq_c_res']:
  frhr_subsample[col] = frhr_subsample[col].astype('float32')
sl.write_feature_stats(
    results_dir, sl.feature_stats(feature_summary, ols_info, male_share)
)
paper_tables.write_ols_temperature_params(
    paper_tables.fit_ols_temperature_params(frhr_subsample),
    data_dir,
    METRIC,
    results_subdir=run_tag,
)
gc.collect()

# ISO-2 -> ISO-3 for the plotly residual map (extra_data/unsd-countries.csv); the map is
# skipped when the reference table is not available (public checkout without extra_data).
_unsd_path = os.path.join(extra_dir, 'unsd-countries.csv')
if fio.file_exists(_unsd_path):
  with fio.open_file(_unsd_path, 'r') as f:
    # keep_default_na=False: Namibia's code is 'NA'.
    ISO2_TO_ISO3 = (
        pd.read_csv(f, keep_default_na=False, na_values=[''], dtype=str)
        .set_index('ISO-alpha2 Code')['ISO-alpha3 Code']
        .to_dict()
    )
  # Codes the UNSD table lacks but the map draws (Taiwan); same supplement
  # as publication_figures.ISO2_TO_ISO3_SUPPLEMENT.
  ISO2_TO_ISO3.update({'TW': 'TWN'})
else:
  ISO2_TO_ISO3 = None
  print(f'{_unsd_path} not found: country residual maps will be skipped')

In [ ]:
# ----------------------------------------------------------------------------
# Candidate right-hand sides for each stage live in stage_competition.DEFAULT_SPEC
# (age form, harmonics, latitude forms, temperature forms); plotting groups here.
# ----------------------------------------------------------------------------
SPEC = stage_competition.DEFAULT_SPEC
print('Stage 1 candidates:', dict(SPEC.stage1))
PLOT_GROUPS = [
    None,
    'un_region',
    'gender',
    'lat_group',
    'age_group',
    'country_2_alpha_cd',
]
PLOT_SUFFIX = {
    None: '',
    'un_region': '_un_region',
    'gender': '_gender',
    'lat_group': '_lat_bin',
    'age_group': '_age_group',
    'country_2_alpha_cd': '_country',
}
VALIDATION = [  # (group col, file stem, title, row key, excluded level)
    (
        'country_2_alpha_cd',
        'country',
        'Country-Level',
        'aggregate_R_country',
        None,
    ),
    (
        'un_region',
        'un_region',
        'UN-Region-Level',
        'aggregate_R_un_region',
        'oceania',
    ),
    ('gender', 'gender', 'Gender-Level', 'aggregate_R_gender', 'other'),
    ('lat_group', 'lat_bin', 'Latitude-Bin-Level', 'aggregate_R_lat_bin', None),
    ('age_group', 'age_group', 'Age-Level', 'aggregate_R_age_group', None),
]

In [ ]:
# ----------------------------------------------------------------------------
# One model fit = one call. Resumable: a folder with model-row.csv is reused
# (the CONFIG check above guarantees it was fitted on the same data).
# ----------------------------------------------------------------------------
def _read_csv(path, **kwargs):
  with fio.open_file(path, 'r') as f:
    return pd.read_csv(f, **kwargs)


def _find_completed(formula):
  tag = formula.replace(' ', '')
  for sub in fio.list_dir(results_dir):
    row_file = os.path.join(results_dir, sub, 'model-row.csv')
    if not fio.file_exists(row_file):
      continue
    row = _read_csv(row_file)
    if not row.empty and str(row['formula'].iloc[0]).replace(' ', '') == tag:
      vif_file = os.path.join(results_dir, sub, 'vif.csv')
      vif = _read_csv(vif_file) if fio.file_exists(vif_file) else None
      return row.iloc[0].to_dict(), vif
  return None, None


def _save_fig(fig, path):
  with fio.open_file(path, 'w') as f:
    fig.savefig(f, format='svg', dpi=300)
  plt.close(fig)


def fit_lme_model(
    formula,
    name,
    generate_plots=False,
    only_latitude_plots=False,
    skip_hessian=False,
    save_random_effects=False,
):
  """Fits one candidate, writes its audit folder, returns (result, model_row, vif)."""
  print(f'\n{"=" * 70}\n{name}: {formula}\n{"=" * 70}')
  if not FORCE_CLEAN:
    row, vif = _find_completed(formula)
    if row is not None:
      print('  [model checkpoint] already fitted - skipping')
      return None, row, vif

  output_path = sl.get_create_output_path(formula, results_dir, colname, METRIC)
  model = lme_binned.BinnedMixedLM.from_formula(
      formula, data=frhr_subsample, groups='user_id_n'
  )
  result = model.fit(reml=False, method='lbfgs', skip_hessian=skip_hessian)
  blups = lme_binned.inject_fast_random_effects(result)
  frhr_subsample['fitted_values'] = result.fittedvalues
  frhr_subsample['predicted_val'] = result.predict()

  model_row, vif = sl.extract_model_stats(
      result,
      model_name=name,
      formula=formula,
      data=frhr_subsample,
      target_col=colname_c,
      scaling_factors=scaling_factors,
  )

  with fio.open_file(output_path + 'vif.csv', 'w') as f:
    vif.to_csv(f, index=False)
  ci = result.conf_int()
  coef_df = pd.DataFrame({
      'Coef.': result.params,
      'Std.Err.': result.bse,
      'z': result.tvalues,
      'P>|z|': result.pvalues,
      '[0.025': ci[0],
      '0.975]': ci[1],
  })
  with fio.open_file(output_path + 'model_coefficients.csv', 'w') as f:
    coef_df.to_csv(f, index=True)
  cov_df = pd.DataFrame(
      result.cov_params(), index=result.param_names, columns=result.param_names
  )
  with fio.open_file(output_path + 'cov_params.csv', 'w') as f:
    cov_df.to_csv(f, index=True)
  if save_random_effects:
    re_df = pd.DataFrame(
        {'user_id_n': blups.index, 'random_intercept': blups.values}
    ).merge(user_mapping_df, on='user_id_n')[['user_id', 'random_intercept']]
    with fio.open_file(output_path + 'random_effects.parquet', 'wb') as f:
      re_df.to_parquet(f, engine='pyarrow', compression='snappy')

  if generate_plots:
    fig_re, fig_resid = sl.summary_report(result, frhr_subsample, colname_title)
    _save_fig(fig_re, output_path + 'random_effects.svg')
    _save_fig(fig_resid, output_path + 'residuals.svg')
  else:
    sl.compute_r2(result, frhr_subsample)

  if generate_plots or only_latitude_plots:
    # 5 % of users (>= 10k) is plenty for the trend plots.
    n_plot = max(10_000, int(0.05 * frhr_subsample['user_id_n'].nunique()))
    rng = np.random.RandomState(RANDOM_STATE)
    plot_users = rng.choice(
        frhr_subsample['user_id_n'].unique(),
        size=min(n_plot, frhr_subsample['user_id_n'].nunique()),
        replace=False,
    )
    plot_df = frhr_subsample[
        frhr_subsample['user_id_n'].isin(plot_users)
    ].copy()

    groups = ['lat_group'] if only_latitude_plots else PLOT_GROUPS
    modes = [False] if only_latitude_plots else [True, False]
    for group_col in groups:
      for use_re in modes:
        stem = 'prediction_fit' if use_re else 'calibration_fit'
        fig = sl.plot_seasonal_trends(
            plot_df,
            result,
            col=colname_c,
            col_name=colname_title,
            group_col=group_col,
            use_random_effects=use_re,
            boxcox_lambda=optimal_lambda,
            min_users_per_group=MIN_USERS_PER_GROUP,
        )
        _save_fig(fig, f'{output_path}{stem}{PLOT_SUFFIX[group_col]}.svg')

    validations = (
        [v for v in VALIDATION if v[0] == 'lat_group']
        if only_latitude_plots
        else VALIDATION
    )
    for col, stem, title, row_key, exclude in validations:
      v_df = (
          frhr_subsample
          if exclude is None
          else frhr_subsample[frhr_subsample[col] != exclude]
      )
      v_fig, r_val, slope_val, agg_df = sl.save_aggregate_validation(
          v_df,
          result,
          group_col=col,
          target_col=colname_c,
          title_prefix=title,
          boxcox_lambda=optimal_lambda,
      )
      if v_fig is None:
        continue
      model_row[row_key] = r_val
      model_row[f'calib_slope_{col}'] = slope_val
      if col == 'un_region':
        bias = agg_df['actual_mean'] - agg_df['predicted_mean']
        model_row.update(
            max_regional_bias=bias.max(),
            min_regional_bias=bias.min(),
            peak_regional_uncertainty=bias.abs().max(),
            regional_bias_std=bias.std(),
        )
      _save_fig(v_fig, f'{output_path}aggregate_validation_{stem}.svg')
      with fio.open_file(f'{output_path}validation_data_{stem}.csv', 'w') as f:
        agg_df.to_csv(f, index=False)
      if col == 'country_2_alpha_cd' and ISO2_TO_ISO3 is not None:
        agg_df['country_3_alpha_cd'] = agg_df['country_2_alpha_cd'].map(
            ISO2_TO_ISO3
        )
        unmapped = agg_df['country_3_alpha_cd'].isna()
        if unmapped.any():
          codes = sorted(agg_df.loc[unmapped, 'country_2_alpha_cd'].unique())
          print(f'  [maps] dropped unmapped ISO-2 code(s): {codes}')
          agg_df = agg_df[~unmapped]
        fig = sl.plot_global_residuals(
            agg_df,
            iso_col='country_3_alpha_cd',
            obs_col='actual_mean',
            pred_col='predicted_mean',
            is_hrv=(METRIC == 'hrv'),
        )
        with fio.open_file(
            f'{output_path}global_residuals_{METRIC}.html', 'w'
        ) as f:
          f.write(fig.to_html(include_plotlyjs='cdn'))

  with fio.open_file(output_path + 'model-row.csv', 'w') as f:
    pd.DataFrame([model_row]).to_csv(f, index=False)

  plt.close('all')
  frhr_subsample.drop(
      columns=['fitted_values', 'predicted_val'], inplace=True, errors='ignore'
  )
  gc.collect()
  return result, model_row, vif

In [ ]:
# ----------------------------------------------------------------------------
# Stage driver (stage_competition.run_competition) with checkpointing: resumes
# from the last completed stage; STAGE_OVERRIDES force a counterfactual path.
# ----------------------------------------------------------------------------
checkpoint_path = os.path.join(results_dir, 'checkpoint_state.json')
if FORCE_CLEAN:
  fio.remove_file(checkpoint_path)
state = lme_binned.load_checkpoint(
    checkpoint_path
) or stage_competition.new_state(STAGE_OVERRIDES)

state, final_df = stage_competition.run_competition(
    fit_lme_model,
    colname_c,
    state,
    spec=SPEC,
    max_stage=MAX_STAGE,
    overrides=STAGE_OVERRIDES,
    checkpoint_fn=lambda s: lme_binned.save_checkpoint(checkpoint_path, s),
    fit_kwargs=dict(generate_plots=GENERATE_PLOTS),
    # Stored in the checkpoint: a checkpoint written on a different sample is
    # not resumed.
    data_key=dict(
        n_rows=len(frhr_subsample),
        n_users=int(frhr_subsample['user_id_n'].nunique()),
        subsample_frac=SUBSAMPLE_FRAC,
        thinned=thinned,
    ),
)

results_csv = os.path.join(results_dir, f'lme-models-results-{METRIC}.csv')
with fio.open_file(results_csv, 'w') as f:
  final_df.to_csv(f, index=False)
print(f'\nSaved {len(final_df)} candidate rows to {results_csv}')
display(
    final_df[[
        'stage',
        'candidate',
        'BIC',
        'bic_winner',
        'progressed',
        'forced',
        'on_path',
    ]]
)

In [ ]:
# ----------------------------------------------------------------------------
# Overall winner (lowest BIC across all on-path rows; without STAGE_OVERRIDES
# every row is on-path): REML refit, save, final plots
# ----------------------------------------------------------------------------
best = stage_competition.overall_winner(final_df)
winner_name, winner_formula = best['model'], best['formula']
print(
    f'OVERALL WINNER: {winner_name}\n  {winner_formula}\n  BIC ='
    f' {best["BIC"]:.2f}'
)

# BIC selection uses ML (required for comparing fixed effects); the reported
# model is refitted with REML for unbiased variance components / SEs.
reml_model = lme_binned.BinnedMixedLM.from_formula(
    winner_formula, data=frhr_subsample, groups='user_id_n'
)
reml_res = reml_model.fit(reml=True, method='lbfgs')
lme_binned.inject_fast_random_effects(reml_res)

ci = reml_res.conf_int()
coeff_df = pd.DataFrame({
    'coef': reml_res.params,
    'std_err': reml_res.bse,
    'z': reml_res.tvalues,
    'P_z': reml_res.pvalues,
    'ci_lower': ci[0],
    'ci_upper': ci[1],
})
if METRIC == 'hrv':  # log scale -> percentage change
  for src, dst in [
      ('coef', 'percent_change'),
      ('ci_lower', 'percent_ci_lower'),
      ('ci_upper', 'percent_ci_upper'),
  ]:
    coeff_df[dst] = (np.exp(coeff_df[src]) - 1) * 100
with fio.open_file(
    os.path.join(results_dir, f'model_{METRIC}_coefficients_interpreted.csv'),
    'w',
) as f:
  coeff_df.to_csv(f, index=True)
display(coeff_df)

# Pickle the REML results without the design matrices.
reml_res._cache.clear()
for attr in ['endog', 'exog', 'groups', 'exog_re', 'binned_data']:
  setattr(reml_res.model, attr, None)
reml_res.model.data.frame = reml_res.model.data.frame.iloc[:1]
for attr in ['orig_endog', 'orig_exog', 'endog', 'exog', 'groups']:
  setattr(reml_res.model.data, attr, None)
buf = io.BytesIO()
reml_res.save(buf)
with fio.open_file(
    os.path.join(results_dir, f'model_{METRIC}_winner.pkl'), 'wb'
) as f:
  f.write(buf.getvalue())

# Final diagnostic + calibration plots for the winner (full set), with random effects saved.
FORCE_CLEAN = True  # bypass the model checkpoint so the plots are regenerated
try:
  _, _, winner_vif = fit_lme_model(
      winner_formula,
      f'Final_Winner_{winner_name}',
      generate_plots=True,
      save_random_effects=True,
  )
finally:
  FORCE_CLEAN = False
high = winner_vif[winner_vif['GVIF_Adj'] > 2.0]
print(
    'Collinearity audit:',
    'PASS (all adjusted GVIF <= 2)' if high.empty else f'\n{high}',
)

In [ ]:
# Which candidate is best on each metric (sanity check of the BIC choice).
cols = {
    'BIC': 'min',
    'AIC': 'min',
    'marginal_R2': 'max',
    'conditional_R2': 'max',
    'rmse': 'min',
    'aggregate_R_lat_bin': 'max',
    'aggregate_R_country': 'max',
}
for col, op in cols.items():
  if col in final_df and final_df[col].notna().any():
    idx = final_df[col].idxmin() if op == 'min' else final_df[col].idxmax()
    print(f'{op} {col:20s}: {final_df.loc[idx, "model"]}')

**Next:** run this notebook for the other metric, then regenerate the model-derived
tables and figures with [`03_model_results.ipynb`](03_model_results.ipynb), pointing
`RUN_TAGS` at `RHR_RUN_TAG` / `HRV_RUN_TAG`.